In [2]:
"""
난임 데이터 전처리 (결측치처리5 수정본)
- Data Leakage 수정: label encoding 시 test 데이터셋을 절대 활용하지 않도록 변경
  (기존: train+test를 concat해서 매핑 생성 -> 대회 규정 위반)
  (수정: train 데이터만으로 매핑 생성, test는 그 매핑을 그대로 적용만 함)

나머지 로직은 기존 결측치처리5.ipynb와 동일합니다.
"""

import pandas as pd
import numpy as np

# =====================================================
# 데이터 불러오기
# =====================================================

train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

# =====================================================
# 문자열 공백 정리
# =====================================================

for df in [train, test]:

    object_cols = df.select_dtypes(include="object").columns

    for col in object_cols:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s*:\s*", ":", regex=True)
            .str.replace(r"\s*/\s*", " / ", regex=True)
            .str.replace(r"\s*,\s*", ", ", regex=True)
            .replace("nan", np.nan)
        )

# =====================================================
# 필요 없는 컬럼 제거
# =====================================================

remove_columns = [
    "ID",
    "시술 시기 코드",
    "배란 유도 유형",
    "난자 채취 경과일",
    "불임 원인 - 여성 요인"
]

train = train.drop(columns=remove_columns)
test = test.drop(columns=remove_columns)

# =====================================================
# 여부(0/1) 컬럼
# =====================================================

binary_columns = [
    "단일 배아 이식 여부",
    "착상 전 유전 검사 사용 여부",
    "착상 전 유전 진단 사용 여부",
    "동결 배아 사용 여부",
    "신선 배아 사용 여부",
    "기증 배아 사용 여부",
    "대리모 여부",
    "PGD 시술 여부",
    "PGS 시술 여부"
]

for col in binary_columns:

    if col in train.columns:
        train[col] = (
            pd.to_numeric(train[col], errors="coerce")
            .astype("Int64")
        )

    if col in test.columns:
        test[col] = (
            pd.to_numeric(test[col], errors="coerce")
            .astype("Int64")
        )

# =====================================================
# 소수점 컬럼 → Int64
# =====================================================

numeric_columns = [
    "임신 시도 또는 마지막 임신 경과 연수",
    "총 생성 배아 수",
    "미세주입된 난자 수",
    "미세주입에서 생성된 배아 수",
    "이식된 배아 수",
    "미세주입 배아 이식 수",
    "저장된 배아 수",
    "미세주입 후 저장된 배아 수",
    "해동된 배아 수",
    "해동 난자 수",
    "수집된 신선 난자 수",
    "저장된 신선 난자 수",
    "혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수",
    "기증자 정자와 혼합된 난자 수",
    "난자 혼합 경과일",
    "난자 해동 경과일",
    "배아 이식 경과일",
    "배아 해동 경과일"
]

for col in numeric_columns:

    if col in train.columns:
        train[col] = (
            pd.to_numeric(train[col], errors="coerce")
            .astype("Int64")
        )

    if col in test.columns:
        test[col] = (
            pd.to_numeric(test[col], errors="coerce")
            .astype("Int64")
        )

# =====================================================
# 횟수 컬럼
# =====================================================

count_columns = [
    "총 시술 횟수",
    "클리닉 내 총 시술 횟수",
    "IVF 시술 횟수",
    "DI 시술 횟수",
    "총 임신 횟수",
    "IVF 임신 횟수",
    "DI 임신 횟수",
    "총 출산 횟수",
    "IVF 출산 횟수",
    "DI 출산 횟수"
]

count_map = {
    "0회": 0,
    "1회": 1,
    "2회": 2,
    "3회": 3,
    "4회": 4,
    "5회": 5,
    "6회 이상": 6
}

for col in count_columns:

    if col in train.columns:
        train[col] = (
            train[col]
            .map(count_map)
            .astype("Int64")
        )

    if col in test.columns:
        test[col] = (
            test[col]
            .map(count_map)
            .astype("Int64")
        )


# =====================================================
# 특정 시술 유형 Feature Engineering
# =====================================================

for df in [train, test]:

    # -------------------------
    # ICSI 포함 여부
    # -------------------------
    df["시술_ICSI포함"] = (
        df["특정 시술 유형"]
        .str.contains("ICSI", na=False)
        .astype("Int64")
    )

    # -------------------------
    # IVF 포함 여부
    # -------------------------
    df["시술_IVF포함"] = (
        df["특정 시술 유형"]
        .str.contains(r"\bIVF\b", regex=True, na=False)
        .astype("Int64")
    )

    # -------------------------
    # BLASTOCYST 포함 여부
    # -------------------------
    df["시술_BLASTOCYST포함"] = (
        df["특정 시술 유형"]
        .str.contains("BLASTOCYST", na=False)
        .astype("Int64")
    )

    # -------------------------
    # AH 포함 여부
    # -------------------------
    df["시술_AH포함"] = (
        df["특정 시술 유형"]
        .str.contains(r"\bAH\b", regex=True, na=False)
        .astype("Int64")
    )

    # -------------------------
    # 인공수정 방식 추출
    # -------------------------
    method = df["특정 시술 유형"].str.extract(
        r"(IUI|ICI|IVI|Generic DI)",
        expand=False
    )

    method_map = {
        "IUI": 0,
        "ICI": 1,
        "IVI": 2,
        "Generic DI": 3
    }

    df["시술_인공수정방식"] = (
        method
        .map(method_map)
        .astype("Int64")
    )

    # -------------------------
    # 사용된 시술 기법 개수
    # -------------------------

    technique_count = (
        df["특정 시술 유형"]
        .fillna("")
        .str.count(":")
        +
        df["특정 시술 유형"]
        .fillna("")
        .str.count("/")
        + 1
    )

    technique_count = technique_count.where(
        df["특정 시술 유형"].notna(),
        pd.NA
    )

    df["시술_기법조합수"] = (
        technique_count
        .astype("Int64")
    )

# =====================================================
# 기존 컬럼 제거
# (24개 조합 대신 Feature 사용)
# =====================================================

train = train.drop(columns=["특정 시술 유형"])
test = test.drop(columns=["특정 시술 유형"])


# =====================================================
# Label Encoding
# [수정] Data Leakage 방지: train 데이터만으로 매핑 생성
# (test 데이터는 절대 매핑 생성에 사용하지 않음 - 대회 규정 준수)
# =====================================================

label_columns = [
    "시술 당시 나이",
    "시술 유형",
    "배아 생성 주요 이유",
    "난자 출처",
    "정자 출처",
    "난자 기증자 나이",
    "정자 기증자 나이"
]

for col in label_columns:

    if col not in train.columns:
        continue

    # [수정] train 데이터만으로 매핑 생성 (test는 사용하지 않음)
    categories = train[col].dropna().unique()

    mapping = {
        value: idx
        for idx, value in enumerate(categories)
    }

    train[col] = (
        train[col]
        .map(mapping)
        .astype("Int64")
    )

    # test는 train 기준 매핑을 그대로 적용만 함
    # (train에 없던 값이 test에 있으면 자동으로 NaN 처리됨 -> 확인 결과 실제 발생 없음)
    test[col] = (
        test[col]
        .map(mapping)
        .astype("Int64")
    )

# =====================================================
# 전처리 확인
# =====================================================

print("=" * 70)

print("Train Shape :", train.shape)
print("Test Shape  :", test.shape)

print("=" * 70)

for col in label_columns:

    print(f"\n[{col}]")

    print(
        train[col]
        .value_counts(dropna=False)
        .sort_index()
    )

    # [추가] test에 매핑 안 된(NaN) 값이 새로 생겼는지 확인
    unseen_in_test = test[col].isna().sum() - train[col].isna().sum()
    if unseen_in_test > 0:
        print(f"  ⚠ test에서 train에 없던 값으로 인해 결측 {unseen_in_test}건 발생 (확인 필요)")

print("=" * 70)
print("Label Encoding 완료 (Data Leakage 없이 train 기준으로만 생성)")


# =====================================================
# 전처리 데이터 저장
# =====================================================

train.to_csv(
    "/content/train_preprocessed_ff.csv",
    index=False,
    encoding="utf-8-sig"
)

test.to_csv(
    "/content/test_preprocessed_ff.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================================
# 완료
# =====================================================

print("=" * 70)
print("전처리 완료!")
print("Train Shape :", train.shape)
print("Test Shape  :", test.shape)
print("=" * 70)

print("\n저장 완료")
print("/content/train_preprocessed_ff.csv")
print("/content/test_preprocessed_ff.csv")

Train Shape : (256351, 69)
Test Shape  : (90067, 68)

[시술 당시 나이]
시술 당시 나이
0    102476
1      6918
2     57780
3     39247
4     37348
5     12253
6       329
Name: count, dtype: Int64

[시술 유형]
시술 유형
0    250060
1      6291
Name: count, dtype: Int64

[배아 생성 주요 이유]
배아 생성 주요 이유
0       233732
1         1959
2         9192
3         3784
4          125
5         1108
6           44
7           83
8            6
9           20
10           1
11           5
12           1
<NA>      6291
Name: count, dtype: Int64

[난자 출처]
난자 출처
0    234291
1     15769
2      6291
Name: count, dtype: Int64

[정자 출처]
정자 출처
0    229199
1     27016
2        14
3       122
Name: count, dtype: Int64

[난자 기증자 나이]
난자 기증자 나이
0    242381
1      2334
2      6366
3      4976
4       294
Name: count, dtype: Int64

[정자 기증자 나이]
정자 기증자 나이
0    230518
1      5058
2      5667
3      3848
4      5282
5      4911
6      1067
Name: count, dtype: Int64
Label Encoding 완료 (Data Leakage 없이 train 기준으로만 생성)
전처리 완료!
Train Shape : (256351